In [1]:
# ============================================================
# PIMA INDIANS DIABETES DATASET
# K-MEANS CLUSTERING + MLP CLASSIFICATION
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_name = "pima-indians-diabetes.csv"

columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df = pd.read_csv(file_name, names=columns)

print("Dataset loaded successfully!")
print()


# ============================================================
# 3. DISPLAY FIRST 5 ROWS
# ============================================================

print("First 5 rows:")
display(df.head())


# ============================================================
# 4. DISPLAY LAST 5 ROWS
# ============================================================

print("\nLast 5 rows:")
display(df.tail())


# ============================================================
# 5. DATASET INFORMATION
# ============================================================

print("\nDataset Information:")
print(df.info())


# ============================================================
# 6. DATASET SHAPE
# ============================================================

print("\nDataset Shape:")
print(df.shape)


# ============================================================
# 7. COLUMN NAMES
# ============================================================

print("\nColumn Names:")
print(df.columns.tolist())


# ============================================================
# 8. CHECK MISSING VALUES
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())


# ============================================================
# 9. CHECK DUPLICATE ROWS
# ============================================================

print("\nNumber of Duplicate Rows:")
print(df.duplicated().sum())


# ============================================================
# 10. STATISTICAL SUMMARY
# ============================================================

print("\nStatistical Summary:")
display(df.describe())


# ============================================================
# 11. CHECK TARGET DISTRIBUTION
# ============================================================

print("\nOutcome Distribution:")
print(df["Outcome"].value_counts())


# ============================================================
# 12. PLOT TARGET DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 5))

df["Outcome"].value_counts().sort_index().plot(
    kind="bar"
)

plt.xlabel("Outcome")
plt.ylabel("Number of Patients")
plt.title("Diabetes Outcome Distribution")
plt.xticks(
    [0, 1],
    ["No Diabetes", "Diabetes"],
    rotation=0
)

plt.show()


# ============================================================
# 13. CHECK ZERO VALUES
# ============================================================

zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

print("\nZero values in important columns:")

for column in zero_columns:
    print(
        column,
        ":",
        (df[column] == 0).sum()
    )


# ============================================================
# 14. REPLACE ZERO VALUES WITH NaN
# ============================================================

df[zero_columns] = df[zero_columns].replace(
    0,
    np.nan
)


# ============================================================
# 15. DISPLAY MISSING VALUES AFTER REPLACEMENT
# ============================================================

print("\nMissing values after replacing zeros:")
print(df.isnull().sum())


# ============================================================
# 16. FILL MISSING VALUES USING MEDIAN
# ============================================================

for column in zero_columns:
    df[column] = df[column].fillna(
        df[column].median()
    )


# ============================================================
# 17. CHECK MISSING VALUES AGAIN
# ============================================================

print("\nMissing values after median imputation:")
print(df.isnull().sum())


# ============================================================
# 18. DISPLAY CLEAN DATA
# ============================================================

print("\nCleaned Dataset:")
display(df.head())


# ============================================================
# 19. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(
    "Outcome",
    axis=1
)

y = df["Outcome"]


print("\nFeature shape:")
print(X.shape)

print("\nTarget shape:")
print(y.shape)


# ============================================================
# 20. FEATURE NAMES
# ============================================================

print("\nFeatures:")
print(X.columns.tolist())


# ============================================================
# 21. STANDARDIZE FEATURES
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)


# ============================================================
# 22. DISPLAY SCALED DATA
# ============================================================

print("\nScaled Data:")
display(X_scaled.head())


# ============================================================
# 23. K-MEANS CLUSTERING
# ELBOW METHOD
# ============================================================

inertia = []

K_values = range(1, 11)

for k in K_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    inertia.append(
        kmeans.inertia_
    )


# ============================================================
# 24. PLOT ELBOW CURVE
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    K_values,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means Clustering")

plt.xticks(K_values)

plt.grid(True)

plt.show()


# ============================================================
# 25. CREATE K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)


# ============================================================
# 26. FIT K-MEANS
# ============================================================

clusters = kmeans.fit_predict(
    X_scaled
)


# ============================================================
# 27. ADD CLUSTER TO DATAFRAME
# ============================================================

df["Cluster"] = clusters


# ============================================================
# 28. DISPLAY CLUSTER RESULTS
# ============================================================

print("\nDataset with K-Means clusters:")
display(df.head(20))


# ============================================================
# 29. CLUSTER DISTRIBUTION
# ============================================================

print("\nCluster Distribution:")
print(df["Cluster"].value_counts())


# ============================================================
# 30. CLUSTER CENTERS
# ============================================================

cluster_centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=X.columns
)

print("\nCluster Centers:")
display(cluster_centers)


# ============================================================
# 31. COMPARE CLUSTERS WITH ACTUAL OUTCOME
# ============================================================

print("\nK-Means Cluster vs Actual Outcome:")

cluster_comparison = pd.crosstab(
    df["Cluster"],
    df["Outcome"]
)

display(cluster_comparison)


# ============================================================
# 32. VISUALIZE K-MEANS CLUSTERS
# USING GLUCOSE AND BMI
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(
    df["Glucose"],
    df["BMI"],
    c=df["Cluster"],
    alpha=0.7
)

plt.xlabel("Glucose")
plt.ylabel("BMI")
plt.title("K-Means Clusters: Glucose vs BMI")

plt.show()


# ============================================================
# 33. ANOTHER K-MEANS VISUALIZATION
# GLUCOSE VS AGE
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(
    df["Glucose"],
    df["Age"],
    c=df["Cluster"],
    alpha=0.7
)

plt.xlabel("Glucose")
plt.ylabel("Age")
plt.title("K-Means Clusters: Glucose vs Age")

plt.show()


# ============================================================
# 34. PREPARE DATA FOR MLP
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 35. DISPLAY TRAINING AND TESTING SIZE
# ============================================================

print("Training data:")
print(X_train.shape)

print("\nTesting data:")
print(X_test.shape)

print("\nTraining target:")
print(y_train.shape)

print("\nTesting target:")
print(y_test.shape)


# ============================================================
# 36. CREATE MLP CLASSIFIER
# ============================================================

mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=42
)


# ============================================================
# 37. TRAIN MLP
# ============================================================

mlp.fit(
    X_train,
    y_train
)

print("\nMLP model trained successfully!")


# ============================================================
# 38. MAKE PREDICTIONS
# ============================================================

y_pred = mlp.predict(
    X_test
)


# ============================================================
# 39. DISPLAY PREDICTIONS
# ============================================================

print("\nPredictions:")
print(y_pred)


# ============================================================
# 40. CALCULATE ACCURACY
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:")
print(accuracy)


# ============================================================
# 41. CALCULATE PRECISION
# ============================================================

precision = precision_score(
    y_test,
    y_pred
)

print("\nPrecision:")
print(precision)


# ============================================================
# 42. CALCULATE RECALL
# ============================================================

recall = recall_score(
    y_test,
    y_pred
)

print("\nRecall:")
print(recall)


# ============================================================
# 43. CALCULATE F1 SCORE
# ============================================================

f1 = f1_score(
    y_test,
    y_pred
)

print("\nF1 Score:")
print(f1)


# ============================================================
# 44. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No Diabetes",
            "Diabetes"
        ]
    )
)


# ============================================================
# 45. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


# ============================================================
# 46. DISPLAY CONFUSION MATRIX
# ============================================================

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "No Diabetes",
        "Diabetes"
    ]
)

disp.plot()

plt.title("MLP Confusion Matrix")

plt.show()


# ============================================================
# 47. PLOT MLP LOSS
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    mlp.loss_curve_
)

plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.title("MLP Training Loss")

plt.grid(True)

plt.show()


# ============================================================
# 48. ACTUAL VS PREDICTED
# ============================================================

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print("\nActual vs Predicted:")
display(results.head(20))


# ============================================================
# 49. FINAL MODEL PERFORMANCE
# ============================================================

performance = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

print("\nFinal MLP Performance:")
display(performance)


# ============================================================
# 50. FINAL K-MEANS RESULTS
# ============================================================

print("\n========================================")
print("K-MEANS RESULTS")
print("========================================")

print("\nNumber of clusters: 2")

print("\nCluster counts:")
print(df["Cluster"].value_counts())

print("\nCluster vs Actual Diabetes Outcome:")
display(
    pd.crosstab(
        df["Cluster"],
        df["Outcome"]
    )
)


# ============================================================
# 51. FINAL SUMMARY
# ============================================================

print("\n========================================")
print("FINAL SUMMARY")
print("========================================")

print(
    f"MLP Accuracy  : {accuracy:.4f}"
)

print(
    f"MLP Precision : {precision:.4f}"
)

print(
    f"MLP Recall    : {recall:.4f}"
)

print(
    f"MLP F1 Score  : {f1:.4f}"
)

print(
    "\nK-Means was used for unsupervised clustering."
)

print(
    "MLP was used for supervised diabetes classification."
)

print(
    "========================================"
)

FileNotFoundError: [Errno 2] No such file or directory: 'pima-indians-diabetes.csv'